## Librerías y carga de datos
Cargamos el dataset original de 923 startups con 49 columnas desde Kaggle.

In [8]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/raw/startup_data.csv')
print(f'Dataset cargado: {df.shape}')

Dataset cargado: (923, 49)


## Eliminación de columnas irrelevantes
Removemos identificadores, coordenadas, fechas crudas y columnas duplicadas que no aportan valor predictivo.

In [9]:
# Columnas que no aportan información al modelo
cols_drop = [
    'Unnamed: 0', 'Unnamed: 6', 'id', 'name', 
    'object_id', 'zip_code', 'closed_at', 'state_code.1',
    'state_code', 'city', 'latitude', 'longitude',
    'founded_at', 'first_funding_at', 'last_funding_at'
]

df_clean = df.drop(columns=cols_drop)
print(f'Shape después de eliminar columnas: {df_clean.shape}')
print(f'Columnas restantes: {df_clean.columns.tolist()}')

Shape después de eliminar columnas: (923, 34)
Columnas restantes: ['labels', 'age_first_funding_year', 'age_last_funding_year', 'age_first_milestone_year', 'age_last_milestone_year', 'relationships', 'funding_rounds', 'funding_total_usd', 'milestones', 'is_CA', 'is_NY', 'is_MA', 'is_TX', 'is_otherstate', 'category_code', 'is_software', 'is_web', 'is_mobile', 'is_enterprise', 'is_advertising', 'is_gamesvideo', 'is_ecommerce', 'is_biotech', 'is_consulting', 'is_othercategory', 'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC', 'has_roundD', 'avg_participants', 'is_top500', 'status']


De 49 columnas pasamos a 34. Se eliminaron identificadores como `id`, `name` y `object_id`, coordenadas geográficas, fechas crudas y `state_code.1` que era duplicado de `state_code`.

## Manejo de nulos y eliminación de columnas de texto
`category_code` y `status` ya están representados como variables dummy en el dataset, por lo que se eliminan. Los nulos en milestones corresponden a startups sin hitos registrados.

In [10]:
# category_code y status ya están representados como dummies, los eliminamos
df_clean = df_clean.drop(columns=['category_code', 'status'])

# Los nulos en age_first_milestone_year y age_last_milestone_year
# corresponden a startups sin milestones — rellenar con 0
df_clean['age_first_milestone_year'] = df_clean['age_first_milestone_year'].fillna(0)
df_clean['age_last_milestone_year']  = df_clean['age_last_milestone_year'].fillna(0)

print(f'Shape final: {df_clean.shape}')
print(f'\nNulos restantes:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print('\nSin nulos.' if df_clean.isnull().sum().sum() == 0 else '')

Shape final: (923, 32)

Nulos restantes:
Series([], dtype: int64)

Sin nulos.


El dataset queda con 32 columnas y cero valores nulos. Los nulos de `age_first_milestone_year` y `age_last_milestone_year` se imputaron con 0, interpretando que esas startups no registraron milestones.

## Split train/test
Dividimos el dataset antes de cualquier transformación para evitar data leakage. Usamos stratify para mantener la misma proporción de clases en ambos sets.

In [14]:
X = df_clean.drop('labels', axis=1)
y = df_clean['labels']

# Split ANTES de cualquier transformación — regla crítica
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test:  {X_test.shape[0]} muestras')
print(f'Acquired en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Acquired en test:  {y_test.sum()} ({y_test.mean()*100:.1f}%)')

Train: 738 muestras
Test:  185 muestras
Acquired en train: 477 (64.6%)
Acquired en test:  120 (64.9%)


El split estratificado garantiza que tanto train (738 muestras) como test (185 muestras) mantengan la proporción original 65/35, evitando sesgos en la evaluación.

## Normalización y SMOTE
Normalizamos las columnas con escalas muy distintas usando StandardScaler ajustado solo sobre train. Luego aplicamos SMOTE para balancear las clases en el training set.

In [15]:
# Columnas que necesitan normalización por su escala
cols_scale = ['funding_total_usd', 'age_first_funding_year', 
              'age_last_funding_year', 'age_first_milestone_year',
              'age_last_milestone_year', 'avg_participants']

# fit SOLO en train — nunca en todo el dataset
scaler = StandardScaler()
X_train[cols_scale] = scaler.fit_transform(X_train[cols_scale])
X_test[cols_scale]  = scaler.transform(X_test[cols_scale])

# SMOTE solo sobre train
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Train antes de SMOTE:  {y_train.value_counts().to_dict()}')
print(f'Train después de SMOTE: {pd.Series(y_train_bal).value_counts().to_dict()}')
print('\nScaler y datos listos.')

Train antes de SMOTE:  {1: 477, 0: 261}
Train después de SMOTE: {1: 477, 0: 477}

Scaler y datos listos.


SMOTE generó muestras sintéticas de la clase minoritaria (closed) hasta igualar ambas clases en 477 muestras cada una. El test set no se modifica - conserva la distribución real para una evaluación honesta del modelo.

## Guardado de datos procesados
Guardamos los 4 splits y el scaler para que los notebooks siguientes los puedan cargar sin reprocesar.

In [16]:
os.makedirs('../data/processed', exist_ok=True)

pd.DataFrame(X_train_bal, columns=X.columns).to_csv('../data/processed/X_train.csv', index=False)
pd.Series(y_train_bal, name='labels').to_csv('../data/processed/y_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(scaler, '../models/checkpoints/scaler.pkl')

print('Archivos guardados en data/processed/:')
print(f'  X_train.csv — {X_train_bal.shape}')
print(f'  y_train.csv — {len(y_train_bal)} filas')
print(f'  X_test.csv  — {X_test.shape}')
print(f'  y_test.csv  — {len(y_test)} filas')
print('  scaler.pkl  — en models/checkpoints/')

Archivos guardados en data/processed/:
  X_train.csv — (954, 31)
  y_train.csv — 954 filas
  X_test.csv  — (185, 31)
  y_test.csv  — 185 filas
  scaler.pkl  — en models/checkpoints/


Los archivos quedaron listos en `data/processed/`. El train set balanceado tiene 954 muestras (477 por clase) y el test set 185 muestras con distribución real. El scaler se guarda en `models/checkpoints/` para usarlo en la app Streamlit al momento de predecir nuevas startups.

## Conclusiones del preprocesamiento

- Se eliminaron 17 columnas: identificadores, texto, coordenadas y fechas crudas
- Se imputaron con 0 los nulos de `age_first_milestone_year` y `age_last_milestone_year` (startups sin milestones)
- Split estratificado 80/20 tiene, train: 738 muestras, test: 185 muestras
- Se normalizaron 6 columnas numéricas con StandardScaler ajustado solo sobre train
- SMOTE balanceó el train set de 477/261 a 477/477
- El test set conserva la distribución real (64.9% / 35.1%) sin modificaciones